# Phase 3 — fine-tune

Two stages:

```
LSUN Dog  ->  Pokemon base (all 2,119)  ->  per-class
```

The base stage exists because 789 images is thin for a manifold this wide.
Mammalian at 300 kimg straight from dogs gave plausible creatures with a lot of
unresolved anatomy — that is sparse coverage, not a bad init. A base trained on
**all 2,119** images learns how Pokemon anatomy resolves; each class then only
has to learn its flavour, in fewer kimg and with less room to overfit.

**The budget is unchanged** at 33.2 sec/kimg:

| | kimg | hours |
|---|---|---|
| 10 classes x 300, from dogs | 3,000 | 27.7 |
| base 1,000 + 10 x 200 | 3,000 | 27.6 |

The small classes gain most: Amphibian's 71 images currently start from dogs;
with a base they start from a model that has seen every Pokemon.

## Order

**Run step 3 alone in one session** (~9.2 h), save the version, then attach that
output as a Dataset and set `BASE` in step 1 for the per-class runs.

`train()` reports which checkpoint it resumed from every time, so there is never
any doubt about which stage a model came from.

> **Save Version → Save & Run All (Commit)** · **GPU T4 x2** · **Internet On**

## 1. Settings

In [ ]:
KIMG      = 200   # per class, from the base (1.8 h). Use 300 if starting from dogs.
BASE_KIMG = 1000  # the Pokemon base, on all 2,119 images (9.2 h)
GPUS      = 2
BATCH_GPU = 32
FREEZED   = 0     # FreezeD. First knob if ada_p climbs past ~0.7.

# None  -> auto-detect a base trained in THIS session, else fall back to LSUN Dog.
# str   -> path to a base snapshot from a previous session, e.g.
#          "/kaggle/input/pokemon-base/network-snapshot-001000.pkl"
BASE = None

print(f"base {BASE_KIMG} kimg = {BASE_KIMG * 33.2 / 3600:.1f} h")
print(f"each class {KIMG} kimg = {KIMG * 33.2 / 3600:.1f} h")

## 2. Setup

Seven patches for torch 2.x; `patch()` asserts each target exists, so a silent
no-op is impossible. Reasoning in `docs/stylegan.md`. The last two matter only
for `GPUS=2`.

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)
NL = chr(10)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"
    f.write_text(s.replace(old, new))

# Kernels report "Failed!" after building fine.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)" + NL, "")

# TypeError: object.__init__() takes exactly one argument
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# R1 needs grid_sample's second derivative, which torch still lacks.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

# batch_gpu is pinned to mb // 8 (NVlabs' rig, not ours). Default unchanged.
patch("train.py", "args.batch_gpu = spec.mb // spec.ref_gpus",
      "args.batch_gpu = int(os.environ.get('BATCH_GPU', spec.mb // spec.ref_gpus))")

# Multi-GPU only: ranks disagree on noise_const. Sync once from rank 0.
patch("training/training_loop.py",
      "    # Print network summary tables.",
      NL.join(["    if num_gpus > 1:",
               "        torch.cuda.set_device(device)",
               "        for _m in [G, D, G_ema]:",
               "            for _, _t in misc.named_params_and_buffers(_m):",
               "                torch.distributed.broadcast(_t, src=0)",
               "",
               "    # Print network summary tables."]))

DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

NOISE = ("conv2d_gradfix", "Grad strides", "grad.sizes()", "bucket_view.sizes()")

def _snapshot(tag):
    """Newest snapshot from a finished run, or None."""
    dirs = sorted(pathlib.Path(f"/kaggle/working/{tag}_run").glob("00000-*"))
    if not dirs:
        return None
    snaps = sorted(dirs[-1].glob("network-snapshot-*.pkl"))
    return str(snaps[-1]) if snaps else None

def _resume_from(explicit):
    """Resolve which checkpoint to start from, and say so out loud."""
    if explicit:
        return explicit, "given explicitly"
    if BASE:
        return BASE, "Pokemon base (BASE setting)"
    auto = _snapshot("base")
    if auto:
        return auto, "Pokemon base (trained this session)"
    return PKL, "LSUN Dog (no base found)"

def _run(tag, source, kimg, resume, why):
    zp = f"/kaggle/working/{tag}.zip"
    if not os.path.exists(zp):
        subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                        f"--source={source}", f"--dest={zp}"], check=True)

    print(f"{tag}: {kimg} kimg, resuming from {why}")
    print(f"        {resume}")
    cmd = [sys.executable, f"{REPO}/train.py",
           f"--outdir=/kaggle/working/{tag}_run", f"--data={zp}", f"--gpus={GPUS}",
           "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
           f"--resume={resume}", "--snap=10", "--metrics=none", f"--kimg={kimg}"]
    if FREEZED:
        cmd.append(f"--freezed={FREEZED}")

    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1,
                         env=dict(os.environ, BATCH_GPU=str(BATCH_GPU)))
    for line in p.stdout:
        if not any(n in line for n in NOISE):
            print(line, end="")
    p.wait()
    print(f"{tag}: exit {p.returncode}, {(time.time()-t0)/3600:.2f} h")

def train_base(kimg=None):
    """Stage 1: one model over ALL classes. dataset_tool rglobs, so pointing at
    the parent folder picks up every class. Unconditional -- no labels."""
    _run("base", DATA, kimg or BASE_KIMG, PKL, "LSUN Dog")

def train(cls, kimg=None, resume=None):
    """Stage 2: one class. Defaults to the Pokemon base when one exists, and
    never to another class's weights."""
    src, why = _resume_from(resume)
    _run(cls, DATA / cls, kimg or KIMG, src, why)

def show(tag):
    """reals, then fakes at start / middle / end, then the ada_p trace."""
    import matplotlib.pyplot as plt
    import PIL.Image

    dirs = sorted(pathlib.Path(f"/kaggle/working/{tag}_run").glob("00000-*"))
    if not dirs:
        print(f"no run found for {tag}")
        return
    run = dirs[-1]
    # fakes_init.png must be excluded: "_" sorts AFTER digits, so a plain glob
    # puts the initialisation last and hides the final grid.
    grids = sorted(run.glob("fakes[0-9]*.png"))
    snaps = sorted(run.glob("network-snapshot-*.pkl"))

    for g in [run / "reals.png", grids[0], grids[len(grids) // 2], grids[-1]]:
        if not g.exists():
            continue
        im = PIL.Image.open(g)
        im.thumbnail((1400, 1400))
        plt.figure(figsize=(16, 16 * im.height / im.width))
        plt.imshow(im); plt.axis("off")
        plt.title(f"{tag} - {g.name}" + ("   <-- REAL images" if g.name == "reals.png" else ""))
        plt.show()

    print(f"{tag}: {len(snaps)} snapshots, "
          f"{sum(s.stat().st_size for s in snaps)/1e9:.1f} GB")
    print(f"  newest: {_snapshot(tag)}")
    ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
    get = lambda t, k: t.get(k, {}).get("mean", 0)
    for t in ticks[::max(1, len(ticks) // 8)]:
        print(f"  kimg {get(t,'Progress/kimg'):6.0f}  G {get(t,'Loss/G/loss'):7.3f}"
              f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")

print("torch", torch.__version__, "|", torch.cuda.device_count(), "gpus | 7 patches applied")

## 3. Pokémon base — all 2,119 images

**~9.2 h. Run this alone in a session.** Everything downstream depends on it, so
look at its grids before spending anything on a class.

`dataset_tool.py` uses `rglob`, so pointing `--source` at the parent folder
sweeps in every class. Unconditional — no labels, no conditional-resume shape
mismatch.

What to look for: coherent anatomy and clean backgrounds. Class identity is *not*
expected here — that is stage 2's job.

In [ ]:
train_base()

In [ ]:
show("base")

## 4. Mammalian — 789 images, 37% of the target

~1.8 h from the base. `train()` prints which checkpoint it used — check it says
**Pokemon base**, not LSUN Dog.

To start from dogs instead: `train("mammalian", kimg=300, resume=PKL)`

In [ ]:
train("mammalian")

In [ ]:
show("mammalian")

## 5. Arthropod — 237 images, 11% of the target

~1.8 h from the base. `train()` prints which checkpoint it used — check it says
**Pokemon base**, not LSUN Dog.

To start from dogs instead: `train("arthropod", kimg=300, resume=PKL)`

In [ ]:
train("arthropod")

In [ ]:
show("arthropod")

## 6. Plant Fungus — 197 images, 9% of the target

~1.8 h from the base. `train()` prints which checkpoint it used — check it says
**Pokemon base**, not LSUN Dog.

To start from dogs instead: `train("plant_fungus", kimg=300, resume=PKL)`

In [ ]:
train("plant_fungus")

In [ ]:
show("plant_fungus")

## Done — then what

1. **Compare against Mammalian-from-dogs.** Fewer melted limbs and fewer
   unresolved faces is the win the base stage is bought for.
2. **Varied, not just sharp.** Sharp but repetitive means memorisation.
3. If `ada_p` passed ~0.7, raise `--target` or try FreezeD.
4. The remaining seven classes cost ~13 h at 200 kimg each.

Snapshots are in this notebook's **Output**, ~350 MB each.